# Week 6 challenge 

## Fine-tuning a Frontier Model

We will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [ ]:
# imports
import sys
import os
import re
import json
from dotenv import load_dotenv
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import login
from openai import OpenAI
from pydantic import BaseModel
from typing import Optional, Self

sys.path.append('..') # Add the parent directory to the Python path so that we can import Ed's pricer module
from pricer.evaluator import evaluate

In [ ]:
class Item(BaseModel):
    """
    An Item is a data-point of a Product with a Price
    """

    title: str
    category: str
    price: float
    full: Optional[str] = None
    weight: Optional[float] = None
    summary: Optional[str] = None
    prompt: Optional[str] = None
    id: Optional[int] = None

In [ ]:
# environment

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
# I will be using Ed's lite dataset

ds = load_dataset("ed-donner/items_lite")

train = [Item.model_validate(row) for row in ds["train"]]
val = [Item.model_validate(row) for row in ds["validation"]]
test = [Item.model_validate(row) for row in ds["test"]]


print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
openai = OpenAI()

# Data size

I will be using 200 datapoints

In [ ]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as the examples are very small, I will be using 200 examples (and 1 epoch)


fine_tune_train = train[:200]
fine_tune_validation = val[:50]

In [ ]:
len(fine_tune_train)

In [ ]:
#Prepare the data for fine-tuning
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [ ]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [ ]:
# Convert the items into jsonl and write them to a file
def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [ ]:
write_jsonl(fine_tune_train, "../jsonl/fine_tune_train.jsonl")

In [ ]:
write_jsonl(fine_tune_validation, "../jsonl/fine_tune_validation.jsonl")

In [ ]:
with open("../jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
with open("../jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
# Now, we can fine-tune the model
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    # I'm using 5 examples per batch and 1 epoch
    hyperparameters={"n_epochs": 1, "batch_size": 5},
    suffix="pricer"
)

In [ ]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
print(fine_tuned_model_name)

In [ ]:
# The prompt
def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [ ]:
# The inference function
def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [ ]:
# Use Ed's evaluator to test the fine-tuned model
evaluate(gpt_4__1_nano_fine_tuned, test)